In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Objective

Identifying and Predicting the Factors That Explain Employee Satisfaction

# Load Data

In [ ]:
file_path = f"data/dataset.csv"

In [ ]:
with open(file = file_path, mode = 'r') as file:
    headers = file.readline().strip()
    
headers

In [ ]:
columns = headers.replace('"', '').split(sep=',')
columns

In [ ]:
df = pd.read_csv(file_path)
df.shape

In [ ]:
df.columns = columns
df.columns

In [ ]:
df.head(5)

In [ ]:
df.info()

# Data Dictionary

- Age: Employee's age
- Attrition: Type of attrition (voluntary resignation, etc.)
- BusinessTravel: Frequency of business travel
- Department: Work department
- DistanceFromHome: Distance from home to work
- Education: Education level
- EducationField: Field of study
- EnvironmentSatisfaction: Satisfaction with the work environment
- Gender: Employee's gender
- JobInvolvement: Job involvement
- JobLevel: Job level
- JobRole: Job role
- JobSatisfaction: Job satisfaction
- MaritalStatus: Marital status
- MonthlyIncome: Monthly income
- NumCompaniesWorked: Number of companies worked for
- Over18: Whether the employee is over 18 years old
- OverTime: Whether the employee works overtime
- PercentSalaryHike: Percentage salary increase
- PerformanceRating: Performance rating
- RelationshipSatisfaction: Satisfaction with workplace relationships
- StandardHours: Standard working hours
- StockOptionLevel: Stock option level
- TotalWorkingYears: Total years worked
- TrainingTimesLastYear: Number of training sessions in the last year
- WorkLifeBalance: Work-life balance
- YearsAtCompany: Years at the company
- YearsInCurrentRole: Years in the current role
- YearsSinceLastPromotion: Years since the last promotion
- YearsWithCurrManager: Years with the current manager
- Employee Source: Employee source (e.g., referral)
- AgeStartedWorking: Age at which they started working

# Target Variable

In [ ]:
df['Attrition'].value_counts()

In [ ]:
# Drop Termination class

df = df[df['Attrition'].isin(['Current employee', 'Voluntary Resignation'])]

- Classe 0: NOT RESIGNED
- Classe 1: RESIGNED

In [ ]:
def enconde_target_vars(x):
    if x == 'Current employee':
        return 0
    else:
        return 1

In [ ]:
df['Attrition'] = df['Attrition'].apply(enconde_target_vars)

In [ ]:
df['Attrition'].value_counts()

In [ ]:
df.head(5)

# Features

In [ ]:
X = df.drop(columns = 'Attrition')
y = df['Attrition']

In [ ]:
X.shape, y.shape

In [ ]:
num_features = X.select_dtypes(include = 'number').columns.to_list()
cat_features = X.select_dtypes(include = 'str').columns.to_list()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

# Pre-Processing

## Numerical Features

In [ ]:
numerical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
    ])

## Categorical Features

In [ ]:
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'missing')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

## Pre-Processing Pipeline

In [ ]:
pre_processor = ColumnTransformer(transformers = [
    ('num', numerical_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

# Complete Pipeline

In [ ]:
clf = Pipeline(
    steps = [('pre', pre_processor), ('classifier', LogisticRegression())]
)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
clf.score(X_test, y_test)

In [ ]:
clf.named_steps

# Extract the Coefficients

In [ ]:
clf.named_steps['classifier'].coef_

In [ ]:
coefficients = clf.named_steps['classifier'].coef_[0]

# Extract Feature Names

In [ ]:
feature_names = clf.named_steps['pre'].get_feature_names_out()

In [ ]:
prefix = "num__"

num_f = [i.removeprefix(prefix) for i in feature_names if i.startswith(prefix)]

In [ ]:
prefix = "cat__"

cat_f = [i.removeprefix(prefix) for i in feature_names if i.startswith(prefix)]

In [ ]:
cols = num_f + cat_f

In [ ]:
len(cols), len(coefficients)

# Feature Importance

In logistic regression, the higher the coefficient, the stronger the positive relationship between that predictor variable and the likelihood of the outcome occurring.

A higher coefficient means a one-unit increase in your variable results in a much larger increase in both the log-odds and the odds of the target event.

For example, if you are predicting the probability of a customer making a purchase, a feature with a coefficient of \(2.5\) has a much stronger positive impact on the likelihood of a purchase than a feature with a coefficient of \(0.5\).

In [ ]:
df_coefs = pd.DataFrame(
    data = {
        'features': cols,
        'coefficients': coefficients
    }
)

In [ ]:
df_coefs = df_coefs.sort_values(by = 'coefficients', axis = 0, ascending=False)
df_coefs.head(10)

# Conclusion

The results indicate that employees most likely to resign are those who travel frequently for work, hold a Technical Degree, joined the company through referrals, are single, and hold the position of Laboratory Technician. These characteristics had the greatest positive influence on the model's prediction of turnover.